In [1]:
#add libraries
%pip install pandas matplotlib numpy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from read_data import save_files

Note: you may need to restart the kernel to use updated packages.


### Read Data

In [2]:
def read_data():
    return [
        pd.read_csv('../data/01-starting_data/development_data/awards_players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/coaches.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/series_post.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams_post.csv')
    ]

awards_players, coaches, players, players_teams, series_post, teams, teams_post = read_data()

### Data Selection

Section where we select relevant data and filter out invariant or irrelevant columns 

In [ ]:
awards_players = awards_players.drop(columns=['lgID']) 
coaches = coaches.drop(columns=['lgID'])
players = players.drop(columns=['firstseason', 'lastseason', 'college', 'collegeOther', 'deathDate'])
players_teams = players_teams.drop(columns=['lgID'])
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser']) #'round', 'series' too ??
teams = teams.drop(columns=['lgID', 'franchID', 'divID', 'arena', 'name', 'seeded', 
                        'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB'])
teams_post = teams_post.drop(columns=['lgID'])

In [4]:
save_files("02-data_selection", 
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"], 
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/02-data_selection
Data coaches saved to ../data/02-data_selection
Data players saved to ../data/02-data_selection
Data players_teams saved to ../data/02-data_selection
Data series_post saved to ../data/02-data_selection
Data teams saved to ../data/02-data_selection
Data teams_post saved to ../data/02-data_selection


### Data Preparation

Section where we treat cases like non-existing values, outliers, etc.

In [5]:
save_files("03-data_preparation",
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/03-data_preparation
Data coaches saved to ../data/03-data_preparation
Data players saved to ../data/03-data_preparation
Data players_teams saved to ../data/03-data_preparation
Data series_post saved to ../data/03-data_preparation
Data teams saved to ../data/03-data_preparation
Data teams_post saved to ../data/03-data_preparation


### Feature Engineering

Section where we create and/or simplify existing metrics to aid the prediction model

In [6]:
# def calculateUPER():
#     data['uPER'] = 

# def calculatePER():
#     data['PER'] = uPER * (lgPace/tmPace) * (15 / lguPER) 

In [7]:
teams['win_loss_ratio'] = teams['won'] / (teams['won'] + teams['lost'])  
teams['confIDbin'] = teams['confID'].apply(lambda x: 1 if x == 'EA' else 0)
#data['next_year_playoff_qualification'] = data.groupby('tmID')['playoff_qualification'].shift(-1)

columns_to_drop = ['lgID', 'rank', 'firstRound', 'semis', 'finals']
teams = teams.drop(columns=columns_to_drop, errors='ignore')

#data.columns

In [8]:
save_files("04-feature_engineering",
            ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
            [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/04-feature_engineering
Data coaches saved to ../data/04-feature_engineering
Data players saved to ../data/04-feature_engineering
Data players_teams saved to ../data/04-feature_engineering
Data series_post saved to ../data/04-feature_engineering
Data teams saved to ../data/04-feature_engineering
Data teams_post saved to ../data/04-feature_engineering


### Data Merging

Section responsible for merging all tables, in a format ready to feed the model

In [9]:
#team metrics

data = pd.merge(teams, teams_post, on=['year', 'tmID'], how='left')

data['playoff_qualification'] = data['playoff'].apply(lambda x: 1.0 if x == 'Y' else 0.0)
data.drop(columns=['playoff'], inplace=True)
data.fillna({'W' : 0, 'L' : 0}, inplace=True)
    
#player stats
player_stats = players_teams.groupby(['tmID', 'year']).agg({
    'points': 'sum',
    'rebounds': 'sum',
    'assists': 'sum',
    'steals': 'sum',
    'blocks': 'sum',
    'turnovers': 'sum'
}).reset_index()

data = pd.merge(data, player_stats, on=['year', 'tmID'], how='left')

coach_stats = coaches.groupby(['year', 'tmID']).agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

data = pd.merge(data, coach_stats, on=["year", "tmID"], how="left")

data.columns
#awards
# awards_count = awards_players.groupby(['playerID', 'year']).size().reset_index(name='num_awards')

# data = pd.merge(data, awards_count, on=["year", "playerID"], how="left")
# data['num_awards'] = data['num_awards'].fillna(0)

Index(['year', 'tmID', 'confID', 'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm',
       'o_3pa', 'o_oreb', 'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to',
       'o_blk', 'o_pts', 'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa',
       'd_oreb', 'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk',
       'd_pts', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
       'won_x', 'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW',
       'confL', 'min', 'attend', 'win_loss_ratio', 'confIDbin', 'W', 'L',
       'playoff_qualification', 'points', 'rebounds', 'assists', 'steals',
       'blocks', 'turnovers', 'won_y', 'lost_y', 'post_wins', 'post_losses'],
      dtype='object')

### Add Year 11 info

In [10]:
def read_data11():
    return [
        pd.read_csv('../data/01-starting_data/challenge/coaches.csv'),
        pd.read_csv('../data/01-starting_data/challenge/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/challenge/teams.csv')
    ]

coaches11, players_teams11, teams11 = read_data11()

coaches11 = coaches11.drop(columns=['lgID'])
players_teams11 = players_teams11.drop(columns=['lgID'])
teams11 = teams11.drop(columns=['lgID', 'franchID', 'arena', 'name'])

# #data11 = pd.merge(teams11, players_teams11, on=['year', 'tmID'], how='left')
# #data11 = pd.merge(data11, coaches11, on=['year', 'tmID'], how='left')

data = pd.concat([data, teams11], ignore_index=True)


In [11]:
def shift_performance(data, columns):
    """
    Shifts the performance metrics for next year
    """
    for column in columns:
        data = data.assign(**{column: data.groupby('tmID')[column].shift(1)})
    return data

data = shift_performance(data, [column for column in data.columns if column not in ['year', 'tmID', 'confID', 'confIDbin', 'playoff_qualification']])
data.fillna(0, inplace=True)


In [12]:
#write df to csv
save_files("05-processed_data", ["processed_data"], [data])

data.fillna(0, inplace=True)

print(f"final data columns {data.columns}")

Data processed_data saved to ../data/05-processed_data
final data columns Index(['year', 'tmID', 'confID', 'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm',
       'o_3pa', 'o_oreb', 'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to',
       'o_blk', 'o_pts', 'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa',
       'd_oreb', 'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk',
       'd_pts', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
       'won_x', 'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW',
       'confL', 'min', 'attend', 'win_loss_ratio', 'confIDbin', 'W', 'L',
       'playoff_qualification', 'points', 'rebounds', 'assists', 'steals',
       'blocks', 'turnovers', 'won_y', 'lost_y', 'post_wins', 'post_losses'],
      dtype='object')


### Model training

In [13]:
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVR

Note: you may need to restart the kernel to use updated packages.


#### Initialization 

In [14]:
#result lists
accuracy_scores = []
error_scores = []

#separate feature and target columns
feature_columns = [col for col in data.columns if col not in ['playoff_qualification', 'tmID', 'confID']]
target_column = 'playoff_qualification'

#Create model
#model = DecisionTreeClassifier(random_state=21) #acc: 0.55 error: 0.59 || acc: 0.59  error: 0.55
model = RandomForestClassifier(random_state=21) #acc: 0.60 error: 0.43 || acc: 0.64  error: 0.44
#model = LogisticRegression(random_state=21)     #acc: 0.64 error: 0.39 || acc: 0.57  error: 0.44
#model = SVR()                                   #acc: 0.62 error: 0.44 || acc: 0.60  error: 0.45

data.columns

Index(['year', 'tmID', 'confID', 'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm',
       'o_3pa', 'o_oreb', 'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to',
       'o_blk', 'o_pts', 'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa',
       'd_oreb', 'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk',
       'd_pts', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
       'won_x', 'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW',
       'confL', 'min', 'attend', 'win_loss_ratio', 'confIDbin', 'W', 'L',
       'playoff_qualification', 'points', 'rebounds', 'assists', 'steals',
       'blocks', 'turnovers', 'won_y', 'lost_y', 'post_wins', 'post_losses'],
      dtype='object')

#### Year training cycle

In [15]:
for year in sorted(data['year'].unique())[1:]:  # Start from the second year (with )
    # Separate train and test data
    year_span = 5
    train_data = data[data['year'] <= year] if year < (year_span - 1) else data[(data['year'] <= year) & (data['year'] >= year - (year_span - 1))]
    test_data = data[data['year'] == year + 1]
    
    # If there's no data for the next year (e.g., last year in the dataset), skip (redundant?)
    if test_data.empty:
        continue

    y_test = test_data[target_column]

    results_df = pd.DataFrame()
    for confID in data['confID'].unique():
        conf_train_data = train_data[train_data['confID'] == confID]
        conf_test_data = test_data[test_data['confID'] == confID]

        
        confIDs = conf_test_data['confID'].values
        teamIDs = conf_test_data['tmID'].values

        conf_X_train = conf_train_data[feature_columns]
        conf_y_train = conf_train_data[target_column]

        conf_X_test = conf_test_data[feature_columns]
        conf_y_test = conf_test_data[target_column]
        
        # Standardize the data
        scaler = StandardScaler()
        conf_X_train = scaler.fit_transform(conf_X_train)
        conf_X_test = scaler.transform(conf_X_test)

        # Train the model
        model.fit(conf_X_train, conf_y_train)

        # Make predictions on the test set
        y_pred_proba = model.predict_proba(conf_X_test)[:,1]
        y_pred_proba_norm = np.round(y_pred_proba * 4 / sum(y_pred_proba),2)  # Normalize to sum to 4

        #results_df['Playoff'] = results_df['Playoff'].apply(lambda x: ) # normalize -> max = 1?

        conf_results_df = pd.DataFrame({
            'tmID': teamIDs,
            'confID': confIDs,
            'Playoff': np.round(y_pred_proba_norm, 2),
        })

        conf_y_pred = np.zeros_like(y_pred_proba_norm)
        top_4_indices = conf_results_df.nlargest(4, 'Playoff').index
        conf_y_pred[top_4_indices] = 1

        conf_results_df['Label'] = conf_y_pred

        results_df = pd.concat([results_df, conf_results_df])
        results_df.sort_values(by='tmID', ascending=True, inplace=True)


    y_pred = results_df['Label']
    y_pred_proba_norm = results_df['Playoff']

    if (year == 10):
        #results_df = results_df.drop(columns=['Label', 'confID'])
        results_df.to_csv('../data/06-results/results.csv', index=False)
    # Calculate accuracy and error

    if year < 10:
        accuracy_scores.append(round(accuracy_score(y_test, y_pred),2))

        print(f"{y_pred_proba_norm} - {y_test.values}")
        error_array = np.abs(y_pred_proba_norm - y_test.values)
        error_score = round(sum(error_array) / len(error_array),2)
        error_scores.append(error_score)

        # Output results for each year
        print(f"Year {year} -> {year + 1}:")
        print(f"Results: \n predict: \n {results_df}\n label: \t {y_pred}\n expected: {y_test.values}\n error: \t {error_array}")
        print(f"  Accuracy: {accuracy_scores[-1]}")
        print(f"  Error: \t {round(sum(error_array), 2)} / {len(error_array)} ({error_score})")
        print("\n")

0    0.52
1    0.56
2    0.49
0    0.48
3    0.43
1    0.97
4    0.55
2    0.37
5    0.60
6    0.44
3    0.32
4    0.32
5    0.83
6    0.18
7    0.53
7    0.41
Name: Playoff, dtype: float64 - [1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1.]
Year 2 -> 3:
Results: 
 predict: 
   tmID confID  Playoff  Label
0  CHA     EA     0.52    1.0
1  CLE     EA     0.56    1.0
2  DET     EA     0.49    0.0
0  HOU     WE     0.48    1.0
3  IND     EA     0.43    0.0
1  LAS     WE     0.97    1.0
4  MIA     EA     0.55    1.0
2  MIN     WE     0.37    0.0
5  NYL     EA     0.60    1.0
6  ORL     EA     0.44    0.0
3  PHO     WE     0.32    0.0
4  POR     WE     0.32    0.0
5  SAC     WE     0.83    1.0
6  SEA     WE     0.18    0.0
7  UTA     WE     0.53    1.0
7  WAS     EA     0.41    0.0
 label: 	 0    1.0
1    1.0
2    0.0
0    1.0
3    0.0
1    1.0
4    1.0
2    0.0
5    1.0
6    0.0
3    0.0
4    0.0
5    1.0
6    0.0
7    1.0
7    0.0
Name: Label, dtype: float64
 expected: [1. 0. 0. 1. 1. 1. 0

### End Results

In [16]:
print(f"Accuracy  {accuracy_scores}")
print(f"Error \t {[float(e) for e in error_scores]}")
print("\nAverage Performance Over All Years:")
print(f"  Average Accuracy: {sum(accuracy_scores) / len(accuracy_scores):.2f}")
print(f"  Average Error: \t {round(sum(error_scores) / len(error_scores),2)}")

Accuracy  [0.62, 0.43, 0.54, 0.69, 0.71, 0.54, 0.71, 0.38]
Error 	 [0.49, 0.47, 0.44, 0.4, 0.41, 0.44, 0.41, 0.53]

Average Performance Over All Years:
  Average Accuracy: 0.58
  Average Error: 	 0.45
